# Gradio 日!

今天我们将使用极其简单的 Gradio 框架来构建用户界面。

准备好享受乐趣吧!

请注意:你的 Gradio 界面可能会显示为"深色模式"或"浅色模式",这取决于你的电脑设置。

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
import gradio as gr # 太棒了!

In [ ]:
# 加载名为 .env 文件中的环境变量
# 打印密钥前缀以帮助调试
# 你可以选择任何你喜欢的提供商 - 或者全部使用 Ollama

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API 密钥已存在,开头为 {openai_api_key[:8]}")
else:
    print("OpenAI API 密钥未设置")
    
if anthropic_api_key:
    print(f"Anthropic API 密钥已存在,开头为 {anthropic_api_key[:7]}")
else:
    print("Anthropic API 密钥未设置")

if google_api_key:
    print(f"Google API 密钥已存在,开头为 {google_api_key[:8]}")
else:
    print("Google API 密钥未设置")

In [ ]:
# 连接到 OpenAI、Anthropic 和 Google;如果不使用 Claude 或 Google,请注释掉相应的行

openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [ ]:
# 让我们用一个简单的函数封装对 GPT-4.1-mini 的调用

system_message = "你是一个有用的助手"

def message_gpt(prompt):
    messages = [{"role": "system", "content": system_message}, {"role": "user", "content": prompt}]
    response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
    return response.choices[0].message.content

In [ ]:
# 这可以揭示"训练截止日期",即训练数据中的最新日期

message_gpt("今天是几号?")

## 用户界面时间到了!

In [ ]:
# 这是一个简单的函数

def shout(text):
    print(f"shout 函数已被调用,输入为 {text}")
    return text.upper()

In [ ]:
shout("hello")

In [ ]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch()

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">注意:使用 Gradio 的 Share 工具</h2>
            <span style="color:#900;">我即将向你展示一种非常酷的方式来与他人分享你的 Gradio 界面。这会将你的 gradio 应用部署为 gradio 网站上的演示,然后允许 gradio 调用 'shout' 函数。它使用一种叫做 'HTTP 隧道' 的高级技术(类似于熟悉它的人所知的 ngrok),许多防病毒程序和企业环境不允许使用这种技术。如果你遇到错误,请直接跳过下一个单元格。<br/>
            </span>
        </td>
    </tr>
</table>

In [ ]:
# 添加 share=True 意味着它可以被公开访问
# 我们下周将介绍一个名为 HuggingFace Spaces 的平台,可以提供更持久的托管
# 注意:某些防病毒软件和企业防火墙可能不允许你使用 share=True。
# 如果你在工作场所或工作网络中,我建议跳过此测试。

gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(share=True)

In [ ]:
# 添加 inbrowser=True 会自动打开一个新的浏览器窗口

gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True)

## 添加身份认证

Gradio 让设置用户名和密码变得非常容易

显然,如果你使用这个功能,请确保密码存放在安全的地方!至少,使用你的 .env 文件

In [ ]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True, auth=("ed", "bananas"))

## 强制深色模式

Gradio 会根据浏览器和电脑的设置以浅色或深色模式显示。有一种方法可以强制 gradio 以深色模式显示,但 Gradio 不建议这样做,因为这应该是用户的偏好(尤其出于无障碍考虑)。但如果你希望强制屏幕显示为深色模式,以下是实现方法。

In [ ]:
# 定义此变量,然后在创建 Interface 时传入 js=force_dark_mode

force_dark_mode = """
function refresh() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}
"""
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never", js=force_dark_mode).launch()

In [ ]:
# 再添加一些内容:

message_input = gr.Textbox(label="你的消息:", info="输入要被大喊出来的消息", lines=7)
message_output = gr.Textbox(label="回复:", lines=8)

view = gr.Interface(
    fn=shout,
    title="Shout", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=["hello", "howdy"], 
    flagging_mode="never"
    )
view.launch()

In [ ]:
# 现在 - 将函数从 "shout" 改为 "message_gpt"

message_input = gr.Textbox(label="你的消息:", info="输入一条发送给 GPT-4.1-mini 的消息", lines=7)
message_output = gr.Textbox(label="回复:", lines=8)

view = gr.Interface(
    fn=message_gpt,
    title="GPT", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=["hello", "howdy"], 
    flagging_mode="never"
    )
view.launch()

In [ ]:
# 让我们使用 Markdown
# 你可能想知道,为什么设置 system_message 会有影响,即使下面的代码中没有引用它?
# 我利用了 system_message 是一个全局变量这一点,它在上面的 message_gpt 函数中被使用(去看看)
# 这不是一个好的软件工程实践,但在 Jupyter Lab 研发期间相当常见!

system_message = "你是一个有用的助手,以 markdown 格式回复但不使用代码块"

message_input = gr.Textbox(label="你的消息:", info="输入一条发送给 GPT-4.1-mini 的消息", lines=7)
message_output = gr.Markdown(label="回复:")

view = gr.Interface(
    fn=message_gpt,
    title="GPT", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "向门外汉解释 Transformer 架构",
        "向有志成为 AI 工程师的人解释 Transformer 架构",
        ], 
    flagging_mode="never"
    )
view.launch()

In [ ]:
# 让我们创建一个流式返回结果的调用
# 如果你想复习一下生成器(yield 关键字),
# 请查看 guides 文件夹中的 Intermediate Python 指南

def stream_gpt(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = openai.chat.completions.create(
        model='gpt-4.1-mini',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
message_input = gr.Textbox(label="你的消息:", info="输入一条发送给 GPT-4.1-mini 的消息", lines=7)
message_output = gr.Markdown(label="回复:")

view = gr.Interface(
    fn=stream_gpt,
    title="GPT", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "向门外汉解释 Transformer 架构",
        "向有志成为 AI 工程师的人解释 Transformer 架构",
        ], 
    flagging_mode="never"
    )
view.launch()

In [ ]:
def stream_claude(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = anthropic.chat.completions.create(
        model='claude-sonnet-4-5-20250929',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
message_input = gr.Textbox(label="你的消息:", info="输入一条发送给 Claude 4.5 Sonnet 的消息", lines=7)
message_output = gr.Markdown(label="回复:")

view = gr.Interface(
    fn=stream_claude,
    title="Claude", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "向门外汉解释 Transformer 架构",
        "向有志成为 AI 工程师的人解释 Transformer 架构",
        ], 
    flagging_mode="never"
    )
view.launch()

## 现在来玩点高级的

如果你不确定生成器和 "yield",请记得查看 Intermediate Python 指南

In [ ]:
def stream_model(prompt, model):
    if model=="GPT":
        result = stream_gpt(prompt)
    elif model=="Claude":
        result = stream_claude(prompt)
    else:
        raise ValueError("未知模型")
    yield from result

In [ ]:
message_input = gr.Textbox(label="你的消息:", info="输入一条发送给 LLM 的消息", lines=7)
model_selector = gr.Dropdown(["GPT", "Claude"], label="选择模型", value="GPT")
message_output = gr.Markdown(label="回复:")

view = gr.Interface(
    fn=stream_model,
    title="LLMs", 
    inputs=[message_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["向门外汉解释 Transformer 架构", "GPT"],
            ["向有志成为 AI 工程师的人解释 Transformer 架构", "Claude"]
        ], 
    flagging_mode="never"
    )
view.launch()

# 构建公司宣传册生成器

现在你已经知道方法了 - 很简单!

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">在阅读接下来的几个单元格之前</h2>
            <span style="color:#900;">
                试着自己来做 - 回到第一周第五天的公司宣传册项目,在末尾添加一个 Gradio 界面。然后再来看解决方案。
            </span>
        </td>
    </tr>
</table>

In [ ]:
from scraper import fetch_website_contents

In [ ]:

# 这又是典型的实验心态 - 我正在改变上面使用的全局变量:

system_message = """
你是一个分析公司网站着陆页内容的助手,
为潜在客户、投资者和招聘对象创建一份关于该公司的简短宣传册。
请以 markdown 格式回复但不使用代码块。
"""

In [ ]:
def stream_brochure(company_name, url, model):
    yield ""
    prompt = f"请为 {company_name} 生成一份公司宣传册。这是他们的着陆页:\n"
    prompt += fetch_website_contents(url)
    if model=="GPT":
        result = stream_gpt(prompt)
    elif model=="Claude":
        result = stream_claude(prompt)
    else:
        raise ValueError("未知模型")
    yield from result

In [ ]:
name_input = gr.Textbox(label="公司名称:")
url_input = gr.Textbox(label="着陆页 URL,包括 http:// 或 https://")
model_selector = gr.Dropdown(["GPT", "Claude"], label="选择模型", value="GPT")
message_output = gr.Markdown(label="回复:")

view = gr.Interface(
    fn=stream_brochure,
    title="宣传册生成器", 
    inputs=[name_input, url_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Hugging Face", "https://huggingface.co", "GPT"],
            ["Edward Donner", "https://edwarddonner.com", "Claude"]
        ], 
    flagging_mode="never"
    )
view.launch()

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Gradio 资源</h2>
            <span style="color:#f71;">如果你想深入学习 Gradio,请查看精彩的文档 - 一个奇妙的"兔子洞"。<br/>
            <a href="https://www.gradio.app/guides/quickstart">https://www.gradio.app/guides/quickstart</a><br/>Gradio 主要为演示、原型和 MVP 设计,但我也经常用它来为高级用户制作内部应用。
            </span>
        </td>
    </tr>
</table>